<a href="https://colab.research.google.com/github/zombie9088/Pytorch_learning/blob/main/first%20CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [3]:
torch.manual_seed(42)

In [4]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
df = pd.read_csv("/content/drive/MyDrive/fmnist/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
X= df.iloc[:,1:].values
y= df.iloc[:, 0].values

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
X_train = X_train/255
X_test= X_test/255

In [10]:
#custom datset class

class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
    self.labels= torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]


In [11]:
#creating train dataset object

train_dataset = CustomDataset(X_train,y_train)


In [12]:
#create test dataset object
test_dataset= CustomDataset(X_test,y_test)

In [13]:
#create train and test dataloader

train_loader= DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader= DataLoader(test_dataset,batch_size=32,shuffle=False)

In [14]:
from torch.nn.modules.batchnorm import BatchNorm1d
#define nn class

class MyNN(nn.Module):
  def __init__(self,input_features):
    super().__init__()

    self.features= nn.Sequential(
        nn.Conv2d(input_features,32,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(32,64,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,10)

    )

  def forward(self,x):
    x= self.features(x)
    x=self.classifier(x)
    return x



In [16]:
#epochs and learning rate

epochs= 100
learning_rate = 0.01

In [17]:
#intiate the model
model = MyNN(1)
model= model.to(device)

#loss function

criterion = nn.CrossEntropyLoss()

#optimiser

optimizer= optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [18]:
#training loop

for epoch in range(epochs):

  total_epoch_loss=0
  for batch_features, batch_labels in train_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    #forward pass
    outputs= model(batch_features)

    #loss
    loss= criterion(outputs,batch_labels)

    optimizer.zero_grad()

    #backprop
    loss.backward()


    #update gradient
    optimizer.step()
    total_epoch_loss= total_epoch_loss+loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {total_epoch_loss/len(train_loader)}")


Epoch 1/100, Loss: 0.6452653192182382
Epoch 2/100, Loss: 0.38536192940672237
Epoch 3/100, Loss: 0.32708115540693206
Epoch 4/100, Loss: 0.2941787290448944
Epoch 5/100, Loss: 0.2670485240506629
Epoch 6/100, Loss: 0.24589766339957714
Epoch 7/100, Loss: 0.23140737534935277
Epoch 8/100, Loss: 0.21292596399908265
Epoch 9/100, Loss: 0.20187225091643632
Epoch 10/100, Loss: 0.18928616901425024
Epoch 11/100, Loss: 0.17558221909652152
Epoch 12/100, Loss: 0.1667904955831667
Epoch 13/100, Loss: 0.15651519148300091
Epoch 14/100, Loss: 0.1482308324125285
Epoch 15/100, Loss: 0.14192126141892125
Epoch 16/100, Loss: 0.13255544724098095
Epoch 17/100, Loss: 0.12412284942126522
Epoch 18/100, Loss: 0.12354091301870843
Epoch 19/100, Loss: 0.11211812342951695
Epoch 20/100, Loss: 0.10871362995977203
Epoch 21/100, Loss: 0.10443778738891706
Epoch 22/100, Loss: 0.09932063815610793
Epoch 23/100, Loss: 0.09439618863367165
Epoch 24/100, Loss: 0.08624023749105011
Epoch 25/100, Loss: 0.08426060017097431
Epoch 26/100, 

In [19]:
#set model to eval mode

model.eval()


MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [20]:
#evaluation code on test data

total=0
correct =0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    outputs= model(batch_features)

    _,predicted = torch.max(outputs,1)
    total+= batch_labels.shape[0]
    correct+= (predicted==batch_labels).sum().item()

print(correct/total)

0.9235


In [21]:
#evaluation code on train data

total=0
correct =0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    outputs= model(batch_features)

    _,predicted = torch.max(outputs,1)
    total+= batch_labels.shape[0]
    correct+= (predicted==batch_labels).sum().item()

print(correct/total)

0.9999583333333333
